# JarvisX LoRA Fine-Tuning Notebook
This notebook trains a LoRA adapter for the JarvisX brain on the 200K autogen dataset directly from Google Colab.


In [ ]:
%%capture
!pip install -q -U transformers peft datasets accelerate bitsandbytes huggingface_hub


In [ ]:
from huggingface_hub import notebook_login

# Paste a write-enabled token when prompted
notebook_login()


In [ ]:
import os

# Optional: set your token explicitly if notebook_login() isn't persisted
# os.environ["HF_TOKEN"] = "hf_your_new_token_here"


In [ ]:
import json
from datasets import Dataset
from huggingface_hub import hf_hub_download

DATASET_ID = "AsithaLKonara/jarvisx-autotrain-200k"
VALIDATION_SPLIT = 0.05
HF_TOKEN = os.environ.get("HF_TOKEN")

train_path = hf_hub_download(
    repo_id=DATASET_ID,
    filename="data/train.jsonl",
    repo_type="dataset",
    token=HF_TOKEN
)

records = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if "messages" in obj:
            records.append({"messages": obj["messages"]})

raw = Dataset.from_list(records)
raw = raw.shuffle(seed=42)
val_count = int(len(raw) * VALIDATION_SPLIT)
train_ds = raw.select(range(val_count, len(raw)))
val_ds = raw.select(range(val_count)) if val_count > 0 else None

print(f"Train examples: {len(train_ds)} | Validation examples: {len(val_ds) if val_ds is not None else 0}")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from transformers import DataCollatorForLanguageModeling

CHAT_TEMPLATE = """{% for message in messages -%}
{{ message['role'].capitalize() }}: {{ message['content'] }}
{% endfor -%}
Assistant:"""

def format_chat(example):
    formatted = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        chat_template=CHAT_TEMPLATE
    )
    return tokenizer(
        formatted,
        max_length=512,
        padding="max_length",
        truncation=True
    )

train_tokenized = train_ds.map(format_chat, batched=False, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(format_chat, batched=False, remove_columns=val_ds.column_names) if val_ds is not None else None

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="jarvisx_lora_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="steps" if val_tokenized is not None else "no",
    eval_steps=500,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator
)

trainer.train()


In [ ]:
from huggingface_hub import HfApi

OUTPUT_DIR = "jarvisx_lora_adapter"
REPO_ID = "AsithaLKonara/jarvis-llm-brain"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Update JarvisX LoRA adapter"
)


## Next Steps
- Restart your Hugging Face Space or Cloud deployment so it loads the new adapter.
- Log key metrics and, if needed, bump `num_train_epochs` or adjust the dataset split for future runs.
